# Soil Evaporation Simulation

This notebook runs the **soil evaporation capacitance (SEC) model** ([Or & Lehman, 2019](https://doi.org/10.1029/2018WR024050)) on post-infiltration moisture from tropical cyclones over the Arabian Peninsula.

**Paper:** [Saleh et al., 2025](https://doi.org/10.1038/s43247-025-02493-w) — *Intensifying tropical cyclones in the Arabian Sea replenish depleting aquifers*

**Workflow (5 steps):**
1. Load soil parameters and infiltration shapefiles
2. Review which SEC parameter row applies to each soil type
3. Run evaporation simulations (one series per year × soil type)
4. Combine soil types into yearly tables with a **Total** column
5. Plot and save the figure

Run from the **repository root** (see README). Most logic lives in `src/sec_evaporation.py`; this notebook explains each step.


## Setup


In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.sec_evaporation import (
    DEFAULT_YEARS,
    SOIL_TYPE_PARAM_MAP,
    load_infiltration_gdfs,
    load_soil_params,
    plot_soil_evaporation,
    prepare_evaporation_data,
    repo_root,
    run_all_simulations,
    summarize_simulation_runs,
    summarize_yearly_totals,
)

BASE_DIR = repo_root(ROOT)
FIGURE_PATH = BASE_DIR / "outputs" / "figures" / "soil_evaporation_by_year.jpeg"
years = DEFAULT_YEARS  # TC years: 2011, 2018, 2020


## Step 1 — Load inputs

- **`soil_params`**: hydraulic properties for each SEC parameter set (`theta_sat`, `theta_crit`, `ks`, `et`, etc.)
- **`gdfs`**: infiltration shapefiles per year and soil type; each polygon has **`s_mm`** (initial moisture, mm) and **`area_m2`** (polygon area)


In [ ]:
soil_params = load_soil_params(BASE_DIR)
gdfs = load_infiltration_gdfs(years, BASE_DIR)
soil_params


In [ ]:
# How many infiltration polygons per soil type and year?
for year in years:
    n_polygons = {soil: len(gdf) for soil, gdf in gdfs[year].items()}
    print(f"{year}: {n_polygons}")


## Step 2 — Soil-type → parameter mapping

Each infiltration soil class uses one row from `soil_param.xlsx` (column `site`).


In [ ]:
pd.DataFrame(
    [{"soil_type": soil, "param_site": site} for soil, site in SOIL_TYPE_PARAM_MAP.items()]
)


## Step 3 — Run SEC simulations

`run_all_simulations` returns **`processed_soil_evap_vol`**: a dictionary with **12 entries** (3 years × 4 soil types).

Each entry is a daily time series of **cumulative evaporation (km³)** for that year and soil type.


In [ ]:
processed_soil_evap_vol = run_all_simulations(gdfs, soil_params, years)
simulation_summary = summarize_simulation_runs(processed_soil_evap_vol)
simulation_summary


## Step 4 — Aggregate by TC year

`prepare_evaporation_data` merges the 12 series into **3 yearly tables** (2011, 2018, 2020).

Each table has columns for the four soil types plus **`Total`** (sum across soils; shorter series are forward-filled to align lengths).


In [ ]:
processed_data = prepare_evaporation_data(processed_soil_evap_vol)
yearly_summary = summarize_yearly_totals(processed_data)
yearly_summary


In [ ]:
# Example: first rows of the 2018 table (TC Mekunu year)
processed_data[2018].head()


## Step 5 — Plot accumulated evaporation

Three panels (a–c) show 2011, 2018, and 2020. The dashed **Total** line is the sum of all soil types.

Y-axis: 0 to max(Total) + 1 km³, with major ticks every 0.5 km³.


In [ ]:
fig = plot_soil_evaporation(processed_data, save_path=FIGURE_PATH, show=True)
FIGURE_PATH
